In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import spacy
import joblib
from datasets import load_dataset
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, confusion_matrix

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
RANDOM_STATE = 42

# Загружаем артефакты обученной модели
artifacts = joblib.load('../models/turn_detector.pkl')
model = artifacts['model']
ohe = artifacts['ohe']
NUMERIC_FEATURES = artifacts['numeric_features']
CATEGORICAL_FEATURES = artifacts['categorical_features']
THRESHOLD = artifacts['threshold']
FEATURE_COLUMNS = artifacts['feature_columns']

print(f"Модель: {type(model).__name__}")
print(f"Порог: {THRESHOLD}")
print(f"Признаков: {len(FEATURE_COLUMNS)}")

# Также понадобятся train-данные MultiWOZ для сравнения распределений
train_df_multiwoz = pd.read_parquet('../data/processed.parquet')
print(f"\nMultiWOZ processed: {len(train_df_multiwoz):,} строк")

Модель: LGBMClassifier
Порог: 0.8699999999999999
Признаков: 22

MultiWOZ processed: 153,983 строк


In [4]:
# ds_daily = load_dataset("daily_dialog", split="test", trust_remote_code=True)
ds_daily = load_dataset("roskoN/dailydialog", split="test")
print(f"Диалогов в DailyDialog test: {len(ds_daily)}")
print(f"Поля: {ds_daily.features}")
print()
print("Пример диалога:")
print(ds_daily[0])

Generating train split:   0%|          | 0/11118 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Диалогов в DailyDialog test: 1000
Поля: {'id': Value(dtype='string', id=None), 'acts': Sequence(feature=Value(dtype='int8', id=None), length=-1, id=None), 'emotions': Sequence(feature=Value(dtype='int8', id=None), length=-1, id=None), 'utterances': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None)}

Пример диалога:
{'id': 'ef13ae7a7502181b0f98b2e5b417aad98550672fada48a4fbd94f4d6aa995ba1_0', 'acts': [3, 2, 3, 4, 3, 4, 3, 2, 3, 4, 2, 3], 'emotions': [0, 6, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0], 'utterances': ['Hey man , you wanna buy some weed ?', 'Some what ?', 'Weed ! You know ? Pot , Ganja , Mary Jane some chronic !', 'Oh , umm , no thanks .', 'I also have blow if you prefer to do a few lines .', 'No , I am ok , really .', 'Come on man ! I even got dope and acid ! Try some !', 'Do you really have all of these drugs ? Where do you get them from ?', 'I got my connections ! Just tell me what you want and I ’ ll even give you one ounce for free .', 'Sounds good ! Let ’ s see , 

In [6]:
def daily_dialog_to_prefixes(dialog_idx, dialog):
    """В DailyDialog нет user/system — берём все реплики как разговорную речь."""
    rows = []
    for turn_idx, utterance in enumerate(dialog['utterances']):  # ← было 'dialog'
        text = utterance.strip()
        if not text:
            continue
        words = text.split()
        n = len(words)
        for i in range(1, n + 1):
            prefix = " ".join(words[:i])
            rows.append({
                "dialogue_id": f"daily_{dialog_idx}",
                "turn_id": turn_idx,
                "prefix_text": prefix,
                "is_end": int(i == n),
            })
    return rows


N_DIALOGS_DAILY = min(len(ds_daily), 1000)

all_rows = []
for i in range(N_DIALOGS_DAILY):
    all_rows.extend(daily_dialog_to_prefixes(i, ds_daily[i]))

df_daily = pd.DataFrame(all_rows)
print(f"Префиксов: {len(df_daily):,}")
print(f"Диалогов:  {df_daily['dialogue_id'].nunique()}")
print(f"Доля is_end=1: {df_daily['is_end'].mean():.3f}")
df_daily.head(10)

Префиксов: 106,631
Диалогов:  1000
Доля is_end=1: 0.073


,dialogue_id,turn_id,prefix_text,is_end
0,daily_0,0,Hey,0
1,daily_0,0,Hey man,0
2,daily_0,0,"Hey man ,",0
3,daily_0,0,"Hey man , you",0
4,daily_0,0,"Hey man , you wanna",0
5,daily_0,0,"Hey man , you wanna buy",0
6,daily_0,0,"Hey man , you wanna buy some",0
7,daily_0,0,"Hey man , you wanna buy some weed",0
8,daily_0,0,"Hey man , you wanna buy some weed ?",1
9,daily_0,1,Some,0


In [7]:
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser", "lemmatizer"])

# Базовые признаки
df_daily['length_words'] = df_daily['prefix_text'].str.split().str.len()
df_daily['length_chars'] = df_daily['prefix_text'].str.len()
df_daily['last_word'] = df_daily['prefix_text'].str.split().str[-1].str.lower()
df_daily['ends_with_punct'] = df_daily['prefix_text'].str.rstrip().str[-1].isin(['.', '?', '!']).astype(int)

QUESTION_WORDS = {"what", "where", "when", "who", "why", "how", "which", "whose"}
df_daily['has_question_word'] = df_daily['prefix_text'].str.lower().apply(
    lambda t: int(any(w in t.split() for w in QUESTION_WORDS))
)

# POS-тег через spaCy
unique_last_words = df_daily['last_word'].unique()
pos_map = {}
for doc in nlp.pipe(unique_last_words, batch_size=500):
    pos_map[doc.text] = doc[0].pos_ if len(doc) > 0 else "X"

df_daily['last_pos'] = df_daily['last_word'].map(pos_map)
df_daily['ends_with_prep'] = (df_daily['last_pos'] == "ADP").astype(int)

print("Фичи рассчитаны")
df_daily.head()

Фичи рассчитаны


,dialogue_id,turn_id,prefix_text,is_end,length_words,length_chars,last_word,ends_with_punct,has_question_word,last_pos,ends_with_prep
0,daily_0,0,Hey,0,1,3,hey,0,0,INTJ,0
1,daily_0,0,Hey man,0,2,7,man,0,0,NOUN,0
2,daily_0,0,"Hey man ,",0,3,9,",",0,0,PUNCT,0
3,daily_0,0,"Hey man , you",0,4,13,you,0,0,PRON,0
4,daily_0,0,"Hey man , you wanna",0,5,19,wanna,0,0,NOUN,0
